<a href="https://colab.research.google.com/github/julia-iza/Trabajo-Practico-Laboratorio/blob/main/inspeccion_y_limpieza_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Inspección y limpieza de los datos — TP Subte

Este notebook va de los **zips originales de SBASE** (unos 12 GB descomprimidos) a los **datos procesados** que usa el análisis: pasajeros por **fecha, hora, línea y estación**, un archivo por año en `data/processed/por_anio/`.

Tiene tres partes:

1. **Inspección.** Recorre todos los archivos originales sin modificarlos y muestra cómo viene cada año: separador, columnas, codificación, formato de las fechas y de la hora, nombres de las líneas y filas con problemas.
2. **Limpieza.** Con lo que mostró la inspección, corrige las fechas, unifica nombres, descarta filas inválidas y suma los pasajeros.
3. **Controles de calidad.** Comprueba que el resultado tenga sentido: totales, días por mes, fechas bien asignadas y estaciones unificadas.

**Antes de correrlo:** los zips tienen que estar en `Mi unidad/TP-Subte/zips/`.

**Dónde quedan los resultados:** los datos procesados, en `data/processed/` (dentro del repositorio clonado en Colab) y en la copia de Drive `Mi unidad/TP-Subte/procesados/`; el reporte de la inspección, en `Mi unidad/TP-Subte/inspeccion_resultado.txt`. Este notebook **no sube archivos a GitHub**: los datos procesados ya están en el repositorio. Si se modifica la limpieza y se regeneran, hay que subirlos a mano a `data/processed/`.

**Tiempo:** unos 10 minutos la inspección y entre 30 y 45 la limpieza la primera vez. Cada año limpio se copia a Drive (`Mi unidad/TP-Subte/procesados/`): si Colab se desconecta, o si se vuelve a correr el notebook, los años ya hechos se recuperan de ahí y la limpieza tarda segundos.

El uso de IA y los prompts están en la última sección.

## 1. Configuración

Lo único que hay que tocar.

- `ANIOS`: los años que se inspeccionan y se procesan.
- `INSPECCIONAR`: con `False` se saltea la inspección (por ejemplo, si solo se quiere rehacer la limpieza). Para la versión que se entrega tiene que quedar en `True`, así se ven sus resultados.
- `REPROCESAR`: con `True` se borran los datos procesados anteriores (en el repositorio clonado y en la copia de Drive) y se limpian todos los años de nuevo. Hace falta cuando cambia la lógica de la limpieza. **Si Colab se desconecta a mitad de camino, cambiarlo a `False` antes de volver a correr**, para que siga desde el último año terminado en lugar de empezar de cero.

In [1]:
import os
import io
import time
import shutil
import zipfile
import subprocess
import datetime as dt
import pandas as pd

REPO_URL = "https://github.com/julia-iza/Trabajo-Practico-Laboratorio.git"
REPO_DIR = "/content/Trabajo-Practico-Laboratorio"
CARPETA_ZIPS = "/content/drive/MyDrive/TP-Subte/zips"
CARPETA_RESPALDO = "/content/drive/MyDrive/TP-Subte/procesados"   # copia de seguridad en Drive
ARCHIVO_REPORTE = "/content/drive/MyDrive/TP-Subte/inspeccion_resultado.txt"

ANIOS = list(range(2015, 2027))
INSPECCIONAR = True
REPROCESAR = True

## 2. Drive y repositorio

Monta Drive (donde están los zips) y clona el repositorio, o trae los últimos cambios si ya estaba clonado en esta sesión. Después se ubica dentro de la carpeta del repositorio: todas las rutas que siguen (`data/...`) son relativas a ella. El repositorio se usa para leer los datos ya procesados; este notebook no sube nada.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}
%cd {REPO_DIR}

Mounted at /content/drive
Cloning into '/content/Trabajo-Practico-Laboratorio'...
remote: Enumerating objects: 97, done.
remote: Counting objects: 100% (43/43), done.
remote: Compressing objects: 100% (43/43), done.
remote: Total 97 (delta 16), reused 0 (delta 0), pack-reused 54 (from 1)
Receiving objects: 100% (97/97), 53.35 MiB | 20.67 MiB/s, done.
Resolving deltas: 100% (27/27), done.
/content/Trabajo-Practico-Laboratorio


---
# Parte 1. Inspección de los datos originales

Antes de limpiar hay que saber cómo vienen los datos. Esta parte **no modifica nada**: lee cada CSV dentro de cada zip, fila por fila, y anota lo que encuentra. Todo lo que muestra queda también en `Mi unidad/TP-Subte/inspeccion_resultado.txt`.

## 3. Funciones de apoyo para la inspección

- `mostrar` imprime un texto y a la vez lo guarda para el reporte; `guardar_reporte` escribe el reporte en el archivo.
- `leer_lineas_contando` lee un archivo línea por línea, tolerando el BOM, las líneas en latin-1 y las líneas enteras entre comillas, y **cuenta** cuántas veces pasa cada cosa.
- `forma` reemplaza los dígitos por `9` para ver el "molde" de una fecha: `01/02/2020` es `99/99/9999` y `3/13/2020` es `9/99/9999`.
- `clasificar_fecha` indica si una fecha tiene una sola lectura posible o es ambigua. En `13/3/2020` el 13 no puede ser un mes, así que seguro es día/mes; en `3/4/2020` no hay forma de saberlo mirando solo ese texto.
- `sumar` y `ordenar` llevan la cuenta de cuántas veces aparece cada valor.

In [3]:
LINEAS_MAX = None   # con un número, se leen solo esas filas por archivo (para pruebas rápidas)

reporte = []

def mostrar(texto=""):
    """Imprime el texto y lo guarda para el reporte final."""
    print(texto)
    reporte.append(str(texto))

def guardar_reporte():
    with open(ARCHIVO_REPORTE, "w", encoding="utf-8") as f:
        f.write("\n".join(reporte) + "\n")

def leer_lineas_contando(archivo_abierto, contadores):
    """Lee línea por línea: saca el BOM, tolera latin-1 y quita comillas envolventes."""
    primera = True
    for crudo in archivo_abierto:
        try:
            linea = crudo.decode("utf-8-sig" if primera else "utf-8")
        except UnicodeDecodeError:
            linea = crudo.decode("latin-1")
            contadores["latin1"] += 1
        primera = False
        linea = linea.strip()
        if linea.startswith('"') and linea.rfind('"') > 0:
            contadores["entre_comillas"] += 1
            if linea.rfind('"') < len(linea) - 1:
                contadores["texto_despues_de_comillas"] += 1   # por ejemplo ';;;;' sobrantes
            linea = linea[1:linea.rfind('"')]
        yield linea

def forma(texto):
    """Reemplaza cada dígito por 9 para ver el formato: '3/13/2020' -> '9/99/9999'."""
    resultado = ""
    for caracter in texto:
        if caracter.isdigit():
            resultado += "9"
        else:
            resultado += caracter
    return resultado

def clasificar_fecha(texto):
    """Indica si la fecha tiene una sola lectura posible o es ambigua."""
    try:
        solo_fecha = texto.split(" ")[0]          # por si viene con hora: '2015-01-01 00:00:00'
        if "/" in solo_fecha:
            a, b = [int(x) for x in solo_fecha.split("/")[:2]]
            if a > 12:
                return "barras: seguro dia/mes"
            if b > 12:
                return "barras: seguro mes/dia"
            return "barras: ambigua"
        if "-" in solo_fecha:
            partes = [int(x) for x in solo_fecha.split("-")[:3]]
            if partes[1] > 12:
                return "guiones: seguro AAAA-DD-MM (invertida)"
            if partes[2] > 12:
                return "guiones: seguro AAAA-MM-DD (normal)"
            return "guiones: ambigua"
    except (ValueError, IndexError):
        pass
    return "otro formato"

def sumar(diccionario, clave):
    diccionario[clave] = diccionario.get(clave, 0) + 1

def ordenar(diccionario, cuantos=10):
    """Las claves más frecuentes, de mayor a menor."""
    pares = sorted(diccionario.items(), key=lambda par: par[1], reverse=True)
    return pares[:cuantos]

print("Funciones listas.")

Funciones listas.


## 4. Inspección de un archivo CSV

Para cada CSV:

1. Muestra el encabezado tal cual viene y detecta el separador (`,` o `;`).
2. Muestra las primeras filas.
3. Recorre todas las filas y anota:
   - cuántas filas tiene y cuántos separadores trae cada una (si varía, hay filas rotas o columnas de más);
   - cuántas líneas tuvieron que leerse como latin-1 y cuántas vienen entre comillas;
   - el molde de las fechas y si son ambiguas;
   - una muestra de fechas repartida a lo largo del archivo, para ver si el formato cambia o si las filas están desordenadas;
   - el largo del texto de la hora (`12:15` contra `12:15:00`);
   - los nombres de las líneas de subte y cuántas filas tiene cada una.

In [4]:
def inspeccionar_csv(archivo_abierto, nombre, anio):
    contadores = {"latin1": 0, "entre_comillas": 0, "texto_despues_de_comillas": 0}
    lineas = leer_lineas_contando(archivo_abierto, contadores)

    try:
        encabezado = next(lineas)
    except StopIteration:
        mostrar("  (archivo vacío)")
        return None

    sep = ";" if encabezado.count(";") > encabezado.count(",") else ","
    columnas = [c.lower().strip().strip('"') for c in encabezado.split(sep)]
    mostrar(f"  Encabezado crudo: {encabezado[:250]!r}")
    mostrar(f"  Separador: {sep!r} | {len(columnas)} columnas: {columnas}")

    pos_fecha = columnas.index("fecha") if "fecha" in columnas else None
    pos_desde = columnas.index("desde") if "desde" in columnas else None
    pos_linea = columnas.index("linea") if "linea" in columnas else None
    if pos_fecha is None or pos_linea is None:
        mostrar("  ATENCIÓN: no hay columna 'fecha' o 'linea' con ese nombre; solo se cuentan filas.")
    posiciones = [p for p in [pos_fecha, pos_desde, pos_linea] if p is not None]
    ultima_pos = max(posiciones) if posiciones else 0

    filas = 0
    ejemplos = []
    separadores_por_fila = {}
    formas_fecha = {}
    tipos_fecha = {}
    ejemplo_por_tipo = {}
    largos_hora = {}
    ejemplo_por_largo = {}
    lineas_subte = {}
    secuencia_fechas = []        # cada fecha distinta, en el orden en que aparece
    filas_sin_fecha = 0
    filas_cortas = 0
    ultima_fecha = None
    inicio = time.time()

    for linea in lineas:
        filas += 1
        if len(ejemplos) < 3:
            ejemplos.append(linea[:250])
        sumar(separadores_por_fila, linea.count(sep))

        if posiciones:
            partes = linea.split(sep, ultima_pos + 1)
            if len(partes) <= ultima_pos:
                filas_cortas += 1
                continue

            if pos_fecha is not None:
                texto = partes[pos_fecha].strip().strip('"')
                if texto == "":
                    filas_sin_fecha += 1
                elif texto != ultima_fecha:          # solo se analiza cuando la fecha cambia
                    ultima_fecha = texto
                    secuencia_fechas.append(texto)
                    sumar(formas_fecha, forma(texto))
                    tipo = clasificar_fecha(texto)
                    sumar(tipos_fecha, tipo)
                    ejemplo_por_tipo.setdefault(tipo, texto)

            if pos_desde is not None:
                hora = partes[pos_desde].strip().strip('"')
                sumar(largos_hora, len(hora))
                ejemplo_por_largo.setdefault(len(hora), hora)

            if pos_linea is not None:
                sumar(lineas_subte, partes[pos_linea].strip().strip('"'))

        if LINEAS_MAX is not None and filas >= LINEAS_MAX:
            mostrar(f"  (cortado en {LINEAS_MAX:,} filas por LINEAS_MAX)")
            break

    mostrar("  Primeras filas:")
    for ejemplo in ejemplos:
        mostrar(f"    {ejemplo!r}")
    mostrar(f"  Filas de datos: {filas:,} | leídas en {time.time() - inicio:.0f} s")
    mostrar(f"  Separadores por fila (cantidad: filas): {ordenar(separadores_por_fila, 5)}")
    mostrar(f"  Líneas en latin-1: {contadores['latin1']} | entre comillas: {contadores['entre_comillas']:,}"
            f" | con texto después de las comillas: {contadores['texto_despues_de_comillas']:,}")
    if filas_cortas:
        mostrar(f"  Filas con menos columnas de las esperadas: {filas_cortas:,}")

    if pos_fecha is not None:
        mostrar(f"  Filas sin fecha: {filas_sin_fecha:,} | cambios de fecha: {len(secuencia_fechas):,}"
                f" | textos de fecha distintos: {len(set(secuencia_fechas)):,}")
        mostrar(f"  Moldes de fecha: {ordenar(formas_fecha)}")
        for tipo, cantidad in ordenar(tipos_fecha):
            mostrar(f"    {tipo}: {cantidad:,} (ej. {ejemplo_por_tipo[tipo]!r})")
        if secuencia_fechas:
            paso = max(1, len(secuencia_fechas) // 12)
            muestra = secuencia_fechas[::paso] + [secuencia_fechas[-1]]
            mostrar(f"  Fechas a lo largo del archivo: {muestra}")

    if pos_desde is not None:
        largos = [(largo, cantidad, ejemplo_por_largo[largo]) for largo, cantidad in ordenar(largos_hora, 5)]
        mostrar(f"  Hora 'desde' (largo, filas, ejemplo): {largos}")

    if pos_linea is not None:
        mostrar(f"  Líneas de subte (nombre: filas): {ordenar(lineas_subte, 15)}")

    return {
        "anio": anio, "archivo": nombre, "separador": sep, "columnas": len(columnas),
        "filas": filas, "filas_sin_fecha": filas_sin_fecha,
        "fechas_ambiguas": sum(c for t, c in tipos_fecha.items() if "ambigua" in t),
        "moldes_fecha": " | ".join(formas_fecha.keys()),
        "latin1": contadores["latin1"],
        "lineas_subte": len(lineas_subte),
    }

print("Función de inspección lista.")

Función de inspección lista.


## 5. Inspección de un zip

Lista todo lo que hay adentro del zip, con su tamaño descomprimido, e inspecciona cada CSV.

El zip de 2015 trae, además de su propio CSV, **copias de otros años** en zips internos. No se inspeccionan: cada año se revisa desde su propio zip.

In [5]:
def inspeccionar_zip(zf, anio):
    resultados = []
    contenido = [info for info in zf.infolist() if not info.is_dir()]
    mostrar(f"Archivos adentro: {len(contenido)}")
    for info in contenido:
        mostrar(f"  - {info.filename} ({info.file_size / 1_000_000:.1f} MB descomprimido)")

    for info in contenido:
        nombre = info.filename
        if nombre.lower().endswith(".csv"):
            mostrar(f"\n>> {nombre}")
            with zf.open(nombre) as archivo:
                resultado = inspeccionar_csv(archivo, nombre, anio)
            if resultado:
                resultados.append(resultado)
        elif nombre.lower().endswith(".zip"):
            mostrar(f"\n>> {nombre}: zip interno (copia de otro año), se saltea.")
        else:
            mostrar(f"\n>> {nombre}: no es un CSV, se saltea.")
    return resultados

print("Función lista.")

Función lista.


## 6. Correr la inspección

Recorre todos los años y guarda el reporte después de cada uno. Es una salida larga: para cada CSV muestra lo descripto en la sección 4. Al final, una tabla resume todos los archivos.

In [6]:
if INSPECCIONAR:
    reporte = []
    resumen_inspeccion = []
    inicio_total = time.time()

    for anio in ANIOS:
        ruta = os.path.join(CARPETA_ZIPS, f"molinetes-{anio}.zip")
        mostrar("\n" + "=" * 90)
        mostrar(f"AÑO {anio}  —  molinetes-{anio}.zip")
        mostrar("=" * 90)
        if not os.path.exists(ruta):
            mostrar("NO ENCONTRADO en Drive, se saltea.")
            continue
        with zipfile.ZipFile(ruta) as zf:
            resumen_inspeccion += inspeccionar_zip(zf, anio)
        guardar_reporte()

    mostrar(f"\nInspección completa en {(time.time() - inicio_total) / 60:.1f} min.")
    guardar_reporte()
else:
    print("Inspección salteada (INSPECCIONAR = False).")


AÑO 2015  —  molinetes-2015.zip
Archivos adentro: 10
  - molinetes-2015.csv (1087.2 MB descomprimido)
  - molinetes-2016.zip (80.7 MB descomprimido)
  - molinetes-2017.zip (121.9 MB descomprimido)
  - molinetes-2018.zip (121.6 MB descomprimido)
  - molinetes-2019.zip (126.0 MB descomprimido)
  - molinetes-2020.zip (37.1 MB descomprimido)
  - molinetes-2022.zip (82.5 MB descomprimido)
  - molinetes-2023.zip (82.7 MB descomprimido)
  - molinetes-2024.zip (78.5 MB descomprimido)
  - molinetes-2025.zip (82.4 MB descomprimido)

>> molinetes-2015.csv
  Encabezado crudo: 'periodo,fecha,desde,hasta,linea,molinete,estacion,pax_pagos,pax_pases_pagos,pax_franq,total'
  Separador: ',' | 11 columnas: ['periodo', 'fecha', 'desde', 'hasta', 'linea', 'molinete', 'estacion', 'pax_pagos', 'pax_pases_pagos', 'pax_franq', 'total']
  Primeras filas:
    '201501,2015-01-01,05:00:00,05:15:00,LINEA_H,LINEA_H_CASEROS_NORTE_TURN01,CASEROS,0.0,0.0,0.0,0.0'
    '201501,2015-01-01,05:30:00,05:45:00,LINEA_A,LINEA_

## 7. Resumen de la inspección

Una fila por cada CSV y un resumen por año. Sirve para comparar los años de un vistazo: si un año tiene otro separador, otra cantidad de columnas, otros moldes de fecha o líneas en latin-1, hay que tratarlo distinto en la limpieza.

In [7]:
if INSPECCIONAR:
    tabla_inspeccion = pd.DataFrame(resumen_inspeccion)
    resumen_por_anio = tabla_inspeccion.groupby("anio").agg(archivos=("archivo", "count"),
                                                            filas=("filas", "sum"),
                                                            filas_sin_fecha=("filas_sin_fecha", "sum"),
                                                            fechas_ambiguas=("fechas_ambiguas", "sum"),
                                                            lineas_latin1=("latin1", "sum"))
    mostrar("\nRESUMEN POR AÑO")
    mostrar(resumen_por_anio.to_string())
    guardar_reporte()
    print(f"\nReporte guardado en {ARCHIVO_REPORTE}")
    display(tabla_inspeccion)


RESUMEN POR AÑO
      archivos     filas  filas_sin_fecha  fechas_ambiguas  lineas_latin1
anio                                                                     
2015         1  10958582                0              144              0
2016         1  11542322                0           836750         116667
2017         1  11938476                0            83901         117169
2018         1  12136454            78207              144              0
2019         1  12662343                0              167              0
2020         1   5781006             1836              192              0
2021         1   8071680                0              204              0
2022        24  12136381                0              312         500289
2023        24  12047696                0              288              0
2024        24  11440440                0             1180          50098
2025        26  13279071                0             4989          30537
2026        21   9951

,anio,archivo,separador,columnas,filas,filas_sin_fecha,fechas_ambiguas,moldes_fecha,latin1,lineas_subte
0,2015,molinetes-2015.csv,",",11,10958582,0,144,9999-99-99,0,12
1,2016,molinetes_2016.csv,",",11,11542322,0,836750,99/99/9999,116667,6
2,2017,molinetes_2017.csv,",",12,11938476,0,83901,99/99/9999,117169,6
3,2018,molinetes-subte-18.csv,",",11,12136454,78207,144,9999-99-99,0,7
4,2019,datahistorica122019.csv,",",11,12662343,0,167,9999-99-99,0,6
...,...,...,...,...,...,...,...,...,...,...
121,2026,Molinetes 2026/202607_PAX15min-ABC-INCLUYEOTRO...,;,10,644764,0,4196,9/9/9999 | 99/9/9999,0,3
122,2026,Molinetes 2026/202607_PAX15min-DEH-INCLUYEOTRO...,;,10,503249,0,3163,9/9/9999 | 99/9/9999,0,3
123,2026,Molinetes 2026/202607_PAX15min-PM-INCLUYEOTROM...,;,10,34432,0,452,9/9/9999 | 99/9/9999,0,1
124,2026,Molinetes 2026/202608_PAX15min-ABC-INCLUYEOTRO...,;,11,889222,0,4190,9/9/9999 | 99/9/9999,0,3


## 8. Qué mostró la inspección y qué decide la limpieza

| Hallazgo | Dónde | Decisión en la limpieza |
| :--- | :--- | :--- |
| Separador `,` o `;` según el año | `,` hasta 2020; `;` desde 2021 | Se detecta en cada archivo: el que más aparece en el encabezado. |
| Las columnas cambian de nombre | `PAX_FREQ` (2016-2017), `pax_TOTAL` (2020-2026), `total` (resto) | Se renombran a `pax_franq` y `total`. |
| Columnas de más | `V1`, `ID_ESTACION`, `periodo`, `origen`; cuatro columnas auxiliares en junio y septiembre de 2022 | Se ignoran: solo se usan fecha, hora, línea, estación y pasajeros. |
| Fecha `AAAA-MM-DD` | 2015, 2018, 2019 | Se lee directamente. |
| Fecha `DD/MM/AAAA`, filas desordenadas | 2016, 2017 | Siempre día/mes: no aparece ninguna fecha que solo pueda ser mes/día. |
| Formato de fecha mezclado dentro del archivo, filas ordenadas | 2020 (día/mes y mes/día), 2021 (día/mes e ISO invertida) | Se elige la lectura más cercana al día siguiente de la fila anterior. |
| Un archivo por mes, filas desordenadas, a veces mes/día | 2022 a 2026 (agosto de 2025 mezcla día/mes y mes/día) | Se elige la lectura que cae en el mes del nombre del archivo. |
| Líneas en latin-1 dentro de archivos UTF-8 | 2016, 2017 y varios meses de 2022, 2024 y 2025 | Cada línea se lee como UTF-8 y, si falla, como latin-1. |
| Línea entera entre comillas, a veces con `;;;;` al final | casi todos los archivos de 2022 a 2026 | Se sacan las comillas y lo que sobre después. |
| Hora sin segundos (`12:15`) | algunos meses de 2025 y 2026 | Solo se usa la hora (el número antes de `:`), así que no afecta. |
| La línea se escribe de tres formas | `A`, `LINEA_A`, `LineaA` | Se unifica como `LineaA`. |
| Estaciones con mayúsculas, acentos o sufijo de línea distintos | `Plaza Miserere` / `PLAZA MISERERE`, `Callao.B` | Mayúsculas, sin acentos y sin sufijo. |
| Nombres de estaciones con caracteres dañados (detectado en el control C) | 2019 a 2022: `SAENZ PEÃ±A`, `SAENZ PE�A`, `SAENZ PE¤A`, `AGÃ¼ERO`, `AG�ERO`, entre otros | Se repara el texto guardado dos veces en UTF-8, se quitan los símbolos sueltos y se corrigen a mano los nombres que quedan incompletos (`SAENZ PEA` → `SAENZ PENA`, `AGERO` → `AGUERO`). |
| La misma estación con otro nombre (detectado en el control C) | Línea E: `Retiro E` hasta 2024 y `Retiro` después | Se corrige a mano: `RETIRO`. |
| Filas sin fecha ni línea, y líneas vacías | 2018, 2020, enero de 2025 | Se descartan. |
| Línea `Prueba` y fechas posteriores al 31/8/2026 | 2025 y 2026 | Se descartan. |
| El zip de 2015 trae copias de otros años | 2015 | Se usa solo su CSV. |

El detalle completo de estos problemas está en `data/README.md`.

---
# Parte 2. Limpieza y agregación

Para cada año:
1. Abre el zip sin descomprimirlo en disco y recorre cada CSV fila por fila.
2. Corrige la fecha según el método que corresponde al año.
3. Unifica los nombres de las líneas y de las estaciones.
4. Suma los pasajeros por fecha, hora, línea y estación.
5. Guarda el resultado en `data/processed/por_anio/molinetes_AAAA.csv.gz`.

## 9. Parámetros de la limpieza

- Dónde se guardan los resultados.
- `ULTIMA_FECHA`: último día con datos reales; los registros posteriores se descartan.
- `TAM_BLOQUE`: cuántas filas se juntan antes de agruparlas. Leer todo junto no entra en la memoria de Colab.
- `RENOMBRAR`: nombres distintos para las mismas columnas.
- `FORMATO_FECHA`: método de fecha de cada archivo anual, según la inspección. Desde 2022 la fecha se resuelve con el mes del nombre del archivo.
- `CONTROLES`: valores calculados antes, para comparar en la Parte 3.

In [8]:
CARPETA_SALIDA = os.path.join("data", "processed", "por_anio")
ARCHIVO_RESUMEN = os.path.join("data", "processed", "resumen_limpieza.csv")
ARCHIVO_ESTACIONES = os.path.join("data", "processed", "estaciones_por_anio.csv")

ULTIMA_FECHA = pd.Timestamp("2026-08-31")   # último día con datos reales
TAM_BLOQUE = 500_000                        # filas que se leen antes de agrupar

COLUMNAS_PAX = ["pax_pagos", "pax_pases_pagos", "pax_franq", "total"]

# Algunos años usan otros nombres para las mismas columnas.
RENOMBRAR = {"pax_freq": "pax_franq", "pax_total": "total"}

# Cómo viene escrita la fecha en los archivos anuales (hallazgos de la inspección):
#   "iso"       -> 2019-01-31, siempre año-mes-día
#   "dia/mes"   -> 31/01/2016, siempre día/mes/año (el archivo NO está ordenado por fecha)
#   "secuencia" -> mezcla día/mes, mes/día e ISO invertida; el archivo SÍ está ordenado,
#                  así que cada fecha ambigua se resuelve mirando la fecha anterior.
# Desde 2022 hay un CSV por mes (por ejemplo 202508_PAX15min-ABC.csv): la fecha se
# resuelve con el año y el mes que figuran al principio del nombre del archivo.
FORMATO_FECHA = {
    2015: "iso", 2018: "iso", 2019: "iso",
    2016: "dia/mes", 2017: "dia/mes",
    2020: "secuencia", 2021: "secuencia",
}

# Valores de control calculados antes con 2019, 2020, 2021, 2025 y 2026 (método anterior).
CONTROLES = {
    "Pasajeros totales": 855_064_366,
    "Fechas distintas": 1_700,
    "Promedio diario feb-2020": 767_900,
    "Promedio diario abr-2020": 28_400,
    "Promedio diario abr-2019": 965_900,
    "Promedio diario ago-2025": 610_600,
    "Promedio diario ago-2026": 586_500,
    "Total ene-2019": 22_849_359,
    "Total abr-2020": 852_910,
}

## 10. Corrección de las fechas

Una fecha como `3/4/2020` puede ser 3 de abril (día/mes) o 4 de marzo (mes/día), y en `2021-01-02` el día y el mes pueden estar invertidos. Por eso:

- `fecha_valida` arma una fecha y devuelve `None` si es imposible (por ejemplo, mes 13).
- `lecturas_posibles` devuelve las dos lecturas de un texto: la **normal** (`DD/MM/AAAA` o `AAAA-MM-DD`) y la **invertida** (`MM/DD/AAAA` o `AAAA-DD-MM`).
- `mas_cercana` elige, entre las lecturas posibles, la más cercana a una fecha de referencia.
- `resolver_por_mes` se usa desde 2022: elige la lectura que cae en el mes del archivo.

Para 2020 y 2021 la referencia es el día siguiente a la fecha de la fila anterior, porque esos archivos están ordenados por fecha. Es una inferencia, no un dato oficial.

In [9]:
def fecha_valida(anio, mes, dia):
    """Devuelve la fecha si existe, o None si es imposible (por ejemplo, mes 13)."""
    try:
        return dt.date(anio, mes, dia)
    except ValueError:
        return None


def lecturas_posibles(texto):
    """
    Devuelve las dos lecturas de una fecha: [normal, invertida].
    Con guiones: normal = AAAA-MM-DD, invertida = AAAA-DD-MM.
    Con barras:  normal = DD/MM/AAAA, invertida = MM/DD/AAAA.
    Una lectura imposible queda como None.
    """
    texto = texto.strip().strip('"').split(" ")[0]   # por si viene con hora
    try:
        if "-" in texto:
            a, b, c = [int(x) for x in texto.split("-")[:3]]
            return [fecha_valida(a, b, c), fecha_valida(a, c, b)]
        if "/" in texto:
            a, b, c = [int(x) for x in texto.split("/")[:3]]
            return [fecha_valida(c, b, a), fecha_valida(c, a, b)]
    except ValueError:
        pass
    return [None, None]


def mas_cercana(candidatas, objetivo):
    """De las fechas posibles, la más cercana al objetivo."""
    validas = [f for f in candidatas if f is not None]
    if not validas:
        return None
    return min(validas, key=lambda f: abs((f - objetivo).days))


def resolver_por_mes(texto, anio, mes):
    """
    Archivos mensuales: se queda con la lectura que cae en el mes del archivo.
    Devuelve (fecha, fuera_de_mes). Si ninguna lectura cae en el mes, usa la más
    cercana a ese mes y lo marca para el resumen.
    """
    candidatas = [f for f in lecturas_posibles(texto) if f is not None]
    for f in candidatas:
        if f.year == anio and f.month == mes:
            return f, False
    return mas_cercana(candidatas, dt.date(anio, mes, 15)), True

## 11. Lectura de los archivos línea por línea

`leer_lineas` hace la misma lectura tolerante que la inspección (BOM, latin-1, comillas), pero sin contar: solo entrega cada línea limpia. Si **cada campo** tiene sus propias comillas (2016), la deja igual, porque `pandas` las interpreta solo.

In [10]:
def leer_lineas(zf, nombre):
    """
    Lee el CSV línea por línea dentro del zip.
    - Saca el BOM de la primera línea.
    - Si una línea no es UTF-8 válido, la lee como latin-1 (pasa en varios años).
    - Si la línea ENTERA está entre comillas (2023 a 2026), saca esas comillas y lo
      que sobre después, como los ';;;;;;;;;' de marzo de 2025. Si cada campo tiene
      sus propias comillas (2016), la deja igual: pandas las entiende solo.
    """
    primera = True
    with zf.open(nombre) as archivo:
        for crudo in archivo:
            try:
                linea = crudo.decode("utf-8-sig" if primera else "utf-8")
            except UnicodeDecodeError:
                linea = crudo.decode("latin-1")
            primera = False
            linea = linea.strip()
            if linea.startswith('"'):
                ultima_comilla = linea.rfind('"')
                interior = linea[1:ultima_comilla]
                if ultima_comilla > 0 and '"' not in interior:
                    linea = interior
            yield linea

## 12. Unificación de líneas y estaciones

- **Línea:** `A`, `LINEA_A` y `LineaA` pasan a ser `LineaA`.
- **Estación**, en este orden:
  1. `reparar_codificacion` arregla los nombres guardados **dos veces** en UTF-8. Por ejemplo, la ñ de `Peña` quedó escrita como `Ã±` (o `ÃÂ±`). Se deshace convirtiendo el texto a bytes como latin-1 y leyéndolo de nuevo como UTF-8; si el nombre estaba bien, la conversión falla y queda igual.
  2. Mayúsculas y sin acentos (`Constitución` → `CONSTITUCION`).
  3. Sin el sufijo de línea (`Callao.B` → `CALLAO`), con los puntos y guiones cambiados por espacios.
  4. Se quitan los símbolos sueltos. En algunos años la ñ o la ü se perdieron y en su lugar quedó un símbolo (`SAENZ PE�A`, `SAENZ PE¤A`, `AG³ERO`): al quitarlo queda `SAENZ PEA` o `AGERO`.
  5. `CORRECCIONES_ESTACIONES` corrige a mano esos nombres incompletos y los cambios de nombre de una misma estación (`RETIRO E` → `RETIRO`). Los casos salen del control C de la Parte 3.

`data/processed/estaciones_por_anio.csv` guarda cómo venía escrita cada estación y a qué nombre se llevó.

In [11]:
def normalizar_linea(serie):
    """'A', 'LINEA_A' y 'LineaA' pasan a ser 'LineaA'. 'PM' y 'LineaPM' pasan a 'LineaPM'."""
    letra = serie.str.upper().str.replace("LINEA", "", regex=False)
    letra = letra.str.replace("_", "", regex=False).str.strip()
    return "Linea" + letra


ACENTOS = {"Á": "A", "É": "E", "Í": "I", "Ó": "O", "Ú": "U", "Ü": "U", "Ñ": "N"}

# Nombres que siguen mal después de la limpieza automática (encontrados con el control C)
CORRECCIONES_ESTACIONES = {
    "SAENZ PEA": "SAENZ PENA",   # la Ñ se perdió en el original (2019 a 2022)
    "AGERO": "AGUERO",           # la Ü se perdió en el original (2019 a 2022)
    "RETIRO E": "RETIRO",        # la línea E la llama "Retiro E" hasta 2024
}


def reparar_codificacion(texto):
    """Repara textos guardados dos veces en UTF-8: 'PeÃ±a' -> 'Peña', 'PeÃÂ±a' -> 'Peña'."""
    for _ in range(2):
        try:
            texto = texto.encode("latin-1").decode("utf-8")
        except (UnicodeEncodeError, UnicodeDecodeError):
            break   # el texto ya estaba bien
    return texto


def normalizar_estacion(serie):
    """Unifica cómo se escribe una estación entre años (ver los pasos en el texto de arriba)."""
    nombre = serie.apply(reparar_codificacion)
    nombre = nombre.str.upper().str.strip()
    for con_acento, sin_acento in ACENTOS.items():
        nombre = nombre.str.replace(con_acento, sin_acento, regex=False)
    nombre = nombre.str.replace(r"\.[A-Z]{1,2}$", "", regex=True)          # sufijo de línea: 'CALLAO.B'
    nombre = nombre.str.replace(".", " ", regex=False).str.replace("-", " ", regex=False)
    nombre = nombre.str.replace(r"[^A-Z0-9 ]", "", regex=True)             # símbolos sueltos
    nombre = nombre.str.split().str.join(" ")                               # espacios dobles
    return nombre.replace(CORRECCIONES_ESTACIONES)

## 13. Procesamiento de un CSV

- `agrupar_bloque` convierte un bloque de líneas en DataFrame, pasa los pasajeros a número, la fecha a `datetime` y saca la hora de la columna `desde`; **descarta las filas sin fecha, línea o estación**, unifica nombres y suma por fecha, hora, línea y estación.
- `procesar_csv` recorre un CSV: detecta el separador, ubica la columna de fecha, corrige cada fecha con el método del año y va juntando bloques. Cada texto de fecha se resuelve una sola vez y se guarda, para no repetir el cálculo en millones de filas. También cuenta las filas vacías, sin fecha y fuera del mes.

In [12]:
def agrupar_bloque(buffer, sep, columnas):
    """Convierte un bloque de líneas en DataFrame y lo agrupa por fecha, hora, línea y estación."""
    df = pd.read_csv(io.StringIO("\n".join(buffer)), sep=sep, names=columnas,
                     header=None, dtype=str, on_bad_lines="skip")
    for columna in COLUMNAS_PAX:
        df[columna] = pd.to_numeric(df[columna], errors="coerce")
    df["fecha"] = pd.to_datetime(df["fecha"], format="%Y-%m-%d", errors="coerce")
    df["hora"] = pd.to_numeric(df["desde"].str.split(":").str[0], errors="coerce")
    df = df[df["fecha"].notna() & df["linea"].notna() & df["estacion"].notna()].copy()

    df["estacion_original"] = df["estacion"].str.strip()
    df["linea"] = normalizar_linea(df["linea"])
    df["estacion"] = normalizar_estacion(df["estacion"])

    agregado = df.groupby(["fecha", "hora", "linea", "estacion"], as_index=False)[COLUMNAS_PAX].sum()
    nombres = df.groupby(["linea", "estacion", "estacion_original"], as_index=False)["total"].sum()
    return agregado, nombres


def procesar_csv(zf, nombre, anio):
    """Recorre un CSV, corrige las fechas y devuelve (agregado, nombres, fila_resumen)."""
    lineas = leer_lineas(zf, nombre)
    encabezado = next(lineas, None)
    if encabezado is None:
        return None

    sep = ";" if encabezado.count(";") > encabezado.count(",") else ","
    columnas = [c.strip().strip('"').lower() for c in encabezado.split(sep)]
    columnas = [RENOMBRAR.get(c, c) for c in columnas]
    pos_fecha = columnas.index("fecha")

    nombre_base = os.path.basename(nombre)
    if anio >= 2022:
        metodo = "mes del archivo"
        anio_archivo, mes_archivo = int(nombre_base[:4]), int(nombre_base[4:6])
    else:
        metodo = FORMATO_FECHA[anio]

    cuenta = {"filas": 0, "filas_sin_columnas": 0, "filas_sin_fecha": 0, "filas_fuera_de_mes": 0}
    ya_resueltas = {}          # texto de fecha -> (fecha en ISO, fuera de mes)
    ultimo_texto, ultima_iso, anterior = None, "", None
    bloques, buffer = [], []

    for linea in lineas:
        partes = linea.split(sep, pos_fecha + 1)
        if len(partes) <= pos_fecha:          # línea vacía o rota
            cuenta["filas_sin_columnas"] += 1
            continue
        cuenta["filas"] += 1
        texto = partes[pos_fecha].strip().strip('"')

        if texto == "":
            iso = ""
        elif metodo == "secuencia":
            # El orden importa: solo se recalcula cuando el texto de la fecha cambia.
            if texto != ultimo_texto:
                if anterior is None:
                    f = lecturas_posibles(texto)[0] or lecturas_posibles(texto)[1]
                else:
                    f = mas_cercana(lecturas_posibles(texto), anterior + dt.timedelta(days=1))
                ultimo_texto = texto
                ultima_iso = f.isoformat() if f else ""
                if f:
                    anterior = f
            iso = ultima_iso
        else:
            # El orden no importa: cada texto se resuelve una sola vez y se guarda.
            if texto not in ya_resueltas:
                if metodo == "mes del archivo":
                    f, fuera = resolver_por_mes(texto, anio_archivo, mes_archivo)
                else:
                    f, fuera = lecturas_posibles(texto)[0], False
                ya_resueltas[texto] = (f.isoformat() if f else "", fuera)
            iso, fuera = ya_resueltas[texto]
            if fuera:
                cuenta["filas_fuera_de_mes"] += 1

        if iso == "":
            cuenta["filas_sin_fecha"] += 1
        partes[pos_fecha] = iso
        buffer.append(sep.join(partes))
        if len(buffer) >= TAM_BLOQUE:
            bloques.append(agrupar_bloque(buffer, sep, columnas))
            buffer = []

    if buffer:
        bloques.append(agrupar_bloque(buffer, sep, columnas))
    if not bloques:
        return None

    agregado = pd.concat([b[0] for b in bloques])
    nombres = pd.concat([b[1] for b in bloques])
    fila_resumen = {"anio": anio, "archivo": nombre, "metodo_fecha": metodo, **cuenta,
                    "pasajeros": int(agregado["total"].sum())}
    return agregado, nombres, fila_resumen

## 14. Procesamiento de un año completo

`procesar_anio` procesa todos los CSV de primer nivel del zip (así ignora las copias que trae el de 2015), los junta, vuelve a agrupar y aplica los filtros finales: saca la línea `Prueba`, las fechas de otro año y las posteriores al 31/8/2026. Guarda el archivo del año, la tabla de estaciones y el resumen.

Si el archivo de un año ya existe, se saltea. Para rehacerlo, hay que borrarlo.

In [13]:
def guardar_reemplazando_anio(nuevo, ruta, anio):
    """Agrega las filas de un año a un CSV de resumen, reemplazando las anteriores de ese año."""
    if os.path.exists(ruta):
        viejo = pd.read_csv(ruta)
        nuevo = pd.concat([viejo[viejo["anio"] != anio], nuevo])
    nuevo.sort_values("anio").to_csv(ruta, index=False)


def procesar_anio(carpeta_zips, anio):
    ruta_zip = os.path.join(carpeta_zips, f"molinetes-{anio}.zip")
    ruta_salida = os.path.join(CARPETA_SALIDA, f"molinetes_{anio}.csv.gz")
    if os.path.exists(ruta_salida):
        print(f"{anio}: ya procesado ({ruta_salida}), se saltea.")
        return
    if not os.path.exists(ruta_zip):
        print(f"{anio}: no se encontró {ruta_zip}, se saltea.")
        return

    print(f"\n{anio}: procesando {ruta_zip}")
    agregados, nombres, resumen = [], [], []
    with zipfile.ZipFile(ruta_zip) as zf:
        for info in zf.infolist():
            # Solo los CSV de primer nivel. El zip de 2015 trae además copias de otros
            # años dentro de zips internos: se ignoran porque cada año tiene su propio zip.
            if not info.filename.lower().endswith(".csv"):
                continue
            resultado = procesar_csv(zf, info.filename, anio)
            if resultado is None:
                continue
            agregados.append(resultado[0])
            nombres.append(resultado[1])
            resumen.append(resultado[2])
            r = resultado[2]
            print(f"  {os.path.basename(info.filename)}: {r['filas']:,} filas, "
                  f"{r['pasajeros']:,} pasajeros, sin fecha: {r['filas_sin_fecha']:,}, "
                  f"fuera del mes: {r['filas_fuera_de_mes']:,}")

    datos = pd.concat(agregados)
    datos = datos.groupby(["fecha", "hora", "linea", "estacion"], as_index=False)[COLUMNAS_PAX].sum()

    # Filtros finales: línea de prueba, fechas de otro año y fechas posteriores al último dato real.
    antes = datos["total"].sum()
    datos = datos[datos["linea"] != "LineaPRUEBA"]
    datos = datos[(datos["fecha"].dt.year == anio) & (datos["fecha"] <= ULTIMA_FECHA)]
    print(f"  Pasajeros descartados por los filtros finales: {antes - datos['total'].sum():,.0f}")

    datos["hora"] = datos["hora"].astype(int)
    for columna in COLUMNAS_PAX:
        datos[columna] = datos[columna].round().astype("int64")
    datos["fecha"] = datos["fecha"].dt.strftime("%Y-%m-%d")

    os.makedirs(CARPETA_SALIDA, exist_ok=True)
    datos.to_csv(ruta_salida, index=False, compression="gzip")
    print(f"  Guardado {ruta_salida}: {len(datos):,} filas, "
          f"{os.path.getsize(ruta_salida) / 1_000_000:.1f} MB")

    tabla_nombres = pd.concat(nombres).groupby(["linea", "estacion", "estacion_original"],
                                               as_index=False)["total"].sum()
    tabla_nombres.insert(0, "anio", anio)
    guardar_reemplazando_anio(tabla_nombres, ARCHIVO_ESTACIONES, anio)
    guardar_reemplazando_anio(pd.DataFrame(resumen), ARCHIVO_RESUMEN, anio)

## 15. Reprocesar o recuperar años ya procesados

- Con `REPROCESAR = True`, borra los resultados anteriores (en el repositorio clonado y en la copia de Drive) para que la sección 16 limpie todos los años de nuevo.
- Con `REPROCESAR = False`, trae de Drive los años ya terminados, para no repetirlos si una corrida anterior se cortó.

In [14]:
if REPROCESAR:
    shutil.rmtree("data/processed", ignore_errors=True)
    shutil.rmtree(CARPETA_RESPALDO, ignore_errors=True)
    print("Resultados anteriores borrados: se limpian todos los años de nuevo.")
os.makedirs(CARPETA_SALIDA, exist_ok=True)

if os.path.exists(CARPETA_RESPALDO):
    for raiz, carpetas, archivos in os.walk(CARPETA_RESPALDO):
        for archivo in archivos:
            origen = os.path.join(raiz, archivo)
            destino = os.path.join("data/processed", os.path.relpath(origen, CARPETA_RESPALDO))
            if not os.path.exists(destino):
                os.makedirs(os.path.dirname(destino), exist_ok=True)
                shutil.copy(origen, destino)
                print("Recuperado:", destino)
print("Ya procesados:", sorted(os.listdir(CARPETA_SALIDA)))

Resultados anteriores borrados: se limpian todos los años de nuevo.
Ya procesados: []


## 16. Procesar todos los años

Para cada CSV original muestra cuántas filas leyó, cuántos pasajeros sumó y cuántas filas no tenían fecha o tenían una fecha fuera del mes. Si el año ya estaba procesado, muestra "ya procesado"; el detalle de esa corrida está en `resumen_limpieza.csv` (control A).

In [15]:
for anio in ANIOS:
    procesar_anio(CARPETA_ZIPS, anio)
    shutil.copytree("data/processed", CARPETA_RESPALDO, dirs_exist_ok=True)
print("\nListo. Respaldo en Drive actualizado.")


2015: procesando /content/drive/MyDrive/TP-Subte/zips/molinetes-2015.zip
  molinetes-2015.csv: 10,958,582 filas, 282,519,120 pasajeros, sin fecha: 0, fuera del mes: 0
  Pasajeros descartados por los filtros finales: 0
  Guardado data/processed/por_anio/molinetes_2015.csv.gz: 522,807 filas, 4.7 MB

2016: procesando /content/drive/MyDrive/TP-Subte/zips/molinetes-2016.zip
  molinetes_2016.csv: 11,542,322 filas, 314,418,191 pasajeros, sin fecha: 0, fuera del mes: 0
  Pasajeros descartados por los filtros finales: 0
  Guardado data/processed/por_anio/molinetes_2016.csv.gz: 545,897 filas, 5.0 MB

2017: procesando /content/drive/MyDrive/TP-Subte/zips/molinetes-2017.zip
  molinetes_2017.csv: 11,938,476 filas, 328,701,725 pasajeros, sin fecha: 0, fuera del mes: 0
  Pasajeros descartados por los filtros finales: 0
  Guardado data/processed/por_anio/molinetes_2017.csv.gz: 547,713 filas, 5.0 MB

2018: procesando /content/drive/MyDrive/TP-Subte/zips/molinetes-2018.zip
  molinetes-subte-18.csv: 12,

---
# Parte 3. Controles de calidad

## 17. Comparación con valores de referencia

Compara con valores calculados antes para 2019, 2020, 2021, 2025 y 2026, y muestra el promedio diario y los días con datos de cada mes de todos los años.

- *Pasajeros totales* debería dar casi igual: la suma no depende de cómo se interpreten las fechas.
- Los promedios de 2025 y 2026 pueden diferir: el método anterior resolvía las fechas mirando el orden del archivo, pero en los archivos mensuales las filas vienen desordenadas.
- En la tabla de días, un mes con menos días que el calendario es un hueco en los datos originales.

In [16]:
def cargar_procesados(anios):
    partes = []
    for anio in anios:
        ruta = os.path.join(CARPETA_SALIDA, f"molinetes_{anio}.csv.gz")
        if os.path.exists(ruta):
            partes.append(pd.read_csv(ruta, parse_dates=["fecha"]))
    return pd.concat(partes) if partes else None


def promedio_diario(df, anio, mes):
    del_mes = df[(df["fecha"].dt.year == anio) & (df["fecha"].dt.month == mes)]
    dias = del_mes["fecha"].nunique()
    return del_mes["total"].sum() / dias if dias else float("nan")


def total_mensual(df, anio, mes):
    return df[(df["fecha"].dt.year == anio) & (df["fecha"].dt.month == mes)]["total"].sum()


def controles():
    print("\n" + "=" * 70)
    print("CONTROLES contra los valores calculados antes (2019, 2020, 2021, 2025, 2026)")
    print("=" * 70)
    df = cargar_procesados([2019, 2020, 2021, 2025, 2026])
    if df is None:
        print("Faltan años para comparar.")
    else:
        obtenidos = {
            "Pasajeros totales": df["total"].sum(),
            "Fechas distintas": df["fecha"].nunique(),
            "Promedio diario feb-2020": promedio_diario(df, 2020, 2),
            "Promedio diario abr-2020": promedio_diario(df, 2020, 4),
            "Promedio diario abr-2019": promedio_diario(df, 2019, 4),
            "Promedio diario ago-2025": promedio_diario(df, 2025, 8),
            "Promedio diario ago-2026": promedio_diario(df, 2026, 8),
            "Total ene-2019": total_mensual(df, 2019, 1),
            "Total abr-2020": total_mensual(df, 2020, 4),
        }
        for nombre, esperado in CONTROLES.items():
            obtenido = obtenidos[nombre]
            diferencia = (obtenido - esperado) / esperado * 100
            marca = "OK" if abs(diferencia) < 0.5 else "REVISAR"
            print(f"{nombre:<26} esperado {esperado:>14,.0f}   obtenido {obtenido:>14,.0f}"
                  f"   ({diferencia:+.2f}%)  {marca}")

    print("\nPROMEDIO DIARIO POR MES (miles de pasajeros) y DÍAS CON DATOS, todos los años")
    df = cargar_procesados(ANIOS)
    if df is None:
        return
    diario = df.groupby("fecha", as_index=False)["total"].sum()
    diario["anio"] = diario["fecha"].dt.year
    diario["mes"] = diario["fecha"].dt.month
    promedio = diario.pivot_table(index="anio", columns="mes", values="total", aggfunc="mean")
    dias = diario.pivot_table(index="anio", columns="mes", values="total", aggfunc="count")
    print((promedio / 1000).round(0).to_string())
    print("\nDías con datos por mes:")
    print(dias.to_string())


controles()


CONTROLES contra los valores calculados antes (2019, 2020, 2021, 2025, 2026)
Pasajeros totales          esperado    855,064,366   obtenido    855,064,601   (+0.00%)  OK
Fechas distintas           esperado          1,700   obtenido          1,700   (+0.00%)  OK
Promedio diario feb-2020   esperado        767,900   obtenido        767,869   (-0.00%)  OK
Promedio diario abr-2020   esperado         28,400   obtenido         28,430   (+0.11%)  OK
Promedio diario abr-2019   esperado        965,900   obtenido        965,933   (+0.00%)  OK
Promedio diario ago-2025   esperado        610,600   obtenido        610,572   (-0.00%)  OK
Promedio diario ago-2026   esperado        586,500   obtenido        586,520   (+0.00%)  OK
Total ene-2019             esperado     22,849,359   obtenido     22,849,359   (+0.00%)  OK
Total abr-2020             esperado        852,910   obtenido        852,910   (+0.00%)  OK

PROMEDIO DIARIO POR MES (miles de pasajeros) y DÍAS CON DATOS, todos los años
mes      1     

## 18. Control A: resumen de la limpieza por año

Filas leídas, filas vacías, sin fecha y fuera del mes, y pasajeros, sumados por año a partir de `resumen_limpieza.csv`.

*Qué esperar:* filas sin fecha cerca de 1.836 en 2020 y 78.207 en 2018, y cerca de 0 en el resto; filas vacías cerca de 82.305 en 2025; filas fuera del mes, pocas.

In [17]:
resumen_limpieza = pd.read_csv(ARCHIVO_RESUMEN)
resumen_limpieza.groupby("anio")[["filas", "filas_sin_columnas", "filas_sin_fecha",
                                  "filas_fuera_de_mes", "pasajeros"]].sum()

,filas,filas_sin_columnas,filas_sin_fecha,filas_fuera_de_mes,pasajeros
anio,,,,,
2015,10958582,0,0,0,282519120
2016,11542322,0,0,0,314418191
2017,11938476,0,0,0,328701725
2018,12136454,0,78207,0,348400114
2019,12662343,0,0,0,341340501
2020,5781006,0,1836,0,77504971
2021,8071680,0,0,0,96077158
2022,12136381,0,0,0,231459457
2023,12047696,0,0,0,243225292


## 19. Control B: prueba del domingo

Los domingos viaja mucha menos gente que un día hábil. Si en algún mes las fechas hubieran quedado mal asignadas (día y mes cruzados), los pasajeros de días hábiles caerían en domingos y la diferencia desaparecería.

Se calcula, para cada mes, el promedio de un domingo dividido por el de un día hábil (los sábados quedan afuera). *Qué esperar:* valores bastante menores a 0,5. **Un mes con un valor cercano a 1 tiene las fechas mezcladas.** Algunos meses de 2020 pueden dar distinto por la cuarentena.

In [18]:
procesados = cargar_procesados(ANIOS)
diario = procesados.groupby("fecha", as_index=False)["total"].sum()
diario["anio"] = diario["fecha"].dt.year
diario["mes"] = diario["fecha"].dt.month
diario["dia_semana"] = diario["fecha"].dt.dayofweek      # 0 = lunes ... 6 = domingo
diario = diario[diario["dia_semana"] != 5].copy()         # se dejan afuera los sábados
diario["tipo"] = diario["dia_semana"].apply(lambda d: "domingo" if d == 6 else "habil")

domingos = diario.pivot_table(index=["anio", "mes"], columns="tipo", values="total", aggfunc="mean")
domingos["domingo_sobre_habil"] = (domingos["domingo"] / domingos["habil"]).round(2)
domingos.sort_values("domingo_sobre_habil", ascending=False).head(15)

,tipo,domingo,habil,domingo_sobre_habil
anio,mes,,,
2022,1,126234.40,394767.904762,0.32
2023,7,264995.40,829500.476190,0.32
2025,7,235254.00,743047.347826,0.32
2024,3,246426.40,761852.952381,0.32
2026,7,205348.75,664255.739130,0.31
2025,12,186712.25,611518.608696,0.31
2022,8,274340.50,871294.956522,0.31
2021,10,153186.20,489497.714286,0.31
2022,2,161345.25,545962.500000,0.30


## 20. Control C: estaciones que no aparecen todos los años

Algunas son reales: estaciones nuevas (Santa Fe de la línea H desde 2016, Facultad de Derecho desde 2018, Correo Central, Catalinas y Retiro de la línea E desde 2019) o el Premetro, que aparece desde 2025. `PRUEBA` aparece porque esta tabla se arma antes del filtro final; en los datos procesados no está.

Pero si dos nombres son **la misma estación** escrita distinto, con huecos que se complementan, quedaron sin unificar y hay que agregarlos a `CORRECCIONES_ESTACIONES` (sección 12). Así se encontraron `SAENZ PENA`, `AGUERO` y `RETIRO`.

In [19]:
estaciones = pd.read_csv(ARCHIVO_ESTACIONES)
presencia = estaciones.pivot_table(index=["linea", "estacion"], columns="anio", values="total", aggfunc="sum")
presencia[presencia.isnull().any(axis=1)]

anio                                  2015       2016       2017       2018  \
linea       estacion                                                          
LineaA      PLAZA DE MAYO        5645166.0  5781632.0  5910212.0  3121170.0   
LineaB      LORIA                      NaN        NaN        NaN        NaN   
LineaC      LORIA                      NaN        NaN        NaN        NaN   
LineaD      LORIA                      NaN        NaN        NaN        NaN   
LineaE      CATALINAS                  NaN        NaN        NaN        NaN   
            CORREO CENTRAL             NaN        NaN        NaN        NaN   
            LORIA                      NaN        NaN        NaN        NaN   
            RETIRO                     NaN        NaN        NaN        NaN   
LineaH      FACULTAD DE DERECHO        NaN        NaN        NaN  1084528.0   
            LORIA                      NaN        NaN        NaN        NaN   
            ND                         NaN        NaN        NaN        NaN   
            SANTA FE                   NaN   961692.0  2709224.0  3508611.0   
LineaPM     COCHEPM                    NaN        NaN        NaN        NaN   
            SAGUIER                    NaN        NaN        NaN        NaN   
LineaPRUEBA PRUEBA                     NaN        NaN        NaN        NaN   

anio                                  2019       2020       2021       2022  \
linea       estacion                                                          
LineaA      PLAZA DE MAYO        4770583.0        NaN   825564.0  3307966.0   
LineaB      LORIA                      NaN        NaN        NaN        NaN   
LineaC      LORIA                      NaN        NaN        NaN        NaN   
LineaD      LORIA                      NaN        NaN        NaN        NaN   
LineaE      CATALINAS             792122.0   280975.0   118914.0   746077.0   
            CORREO CENTRAL       1122694.0   446114.0   440206.0  1355904.0   
            LORIA                      NaN        NaN        NaN        NaN   
            RETIRO                862060.0   517353.0   590500.0  1164307.0   
LineaH      FACULTAD DE DERECHO  1940732.0   369698.0   392611.0  1525411.0   
            LORIA                      NaN        NaN        NaN        NaN   
            ND                         NaN        NaN        NaN        NaN   
            SANTA FE             3600814.0  1003626.0  1406556.0  2540468.0   
LineaPM     COCHEPM                    NaN        NaN        NaN        NaN   
            SAGUIER                    NaN        NaN        NaN        NaN   
LineaPRUEBA PRUEBA                     NaN        NaN        NaN        NaN   

anio                                  2023       2024       2025       2026  
linea       estacion                                                         
LineaA      PLAZA DE MAYO        3823943.0  3574930.0  3398439.0  2051727.0  
LineaB      LORIA                      NaN        NaN       33.0        NaN  
LineaC      LORIA                      NaN        NaN        3.0        NaN  
LineaD      LORIA                      NaN        NaN       14.0        NaN  
LineaE      CATALINAS            1089832.0  1005502.0  1169593.0   838452.0  
            CORREO CENTRAL       1593691.0  1507444.0  1513986.0   976375.0  
            LORIA                      NaN        NaN       12.0        NaN  
            RETIRO               1265466.0  1174755.0  1019575.0   650770.0  
LineaH      FACULTAD DE DERECHO  1627977.0  1308055.0  1478360.0  1033650.0  
            LORIA                      NaN        NaN       18.0        NaN  
            ND                         NaN        NaN        8.0        1.0  
            SANTA FE             2563429.0  2475949.0  2413106.0  1611072.0  
LineaPM     COCHEPM                    NaN        NaN    43880.0   757844.0  
            SAGUIER                    NaN        NaN   113455.0   106535.0  
LineaPRUEBA PRUEBA                     NaN        NaN        1.0

## 21. Control D: total del archivo original contra total procesado

Se suman los pasajeros de los archivos originales de 2019 y 2020 leyéndolos de la forma más simple posible, sin pasar por la limpieza. Si coinciden con los procesados, las filas descartadas no tenían pasajeros (o tenían muy pocos). Se eligen esos dos años porque son archivos que `pandas` lee sin ayuda.

In [20]:
for anio, archivo, columna in [(2019, "datahistorica122019.csv", "total"), (2020, "historico2.csv", "pax_TOTAL")]:
    ruta = os.path.join(CARPETA_ZIPS, f"molinetes-{anio}.zip")
    if not os.path.exists(ruta):
        print(f"{anio}: no está el zip, se saltea.")
        continue
    with zipfile.ZipFile(ruta) as zf:
        original = pd.read_csv(zf.open(archivo), usecols=[columna])
    procesado = pd.read_csv(os.path.join(CARPETA_SALIDA, f"molinetes_{anio}.csv.gz"))
    print(f"{anio}: original {original[columna].sum():,.0f} | procesado {procesado['total'].sum():,.0f}")

2019: original 341,340,501 | procesado 341,340,501
2020: original 77,504,971 | procesado 77,504,971


---
## 22. Guardar este notebook en GitHub

Si se corrió con `REPROCESAR = True`, los datos procesados cambiaron y **hay que subirlos a mano**: descargar de Drive la carpeta `Mi unidad/TP-Subte/procesados/` y, en GitHub, subir los archivos de `por_anio/` a `data/processed/por_anio/` y los dos CSV a `data/processed/` (**Add file > Upload files**). Los archivos nuevos reemplazan a los anteriores.

Después, con todas las salidas a la vista: **Archivo > Guardar una copia en GitHub** → repositorio `julia-iza/Trabajo-Practico-Laboratorio`, rama `main`, ruta `notebooks/inspeccion_y_limpieza.ipynb`, con **Incluir un vínculo a Colab** marcado.

---
## 23. Uso de inteligencia artificial

**Herramienta:** Claude (Anthropic), en una conversación de chat. La IA escribió el código a partir de los prompts de abajo. El grupo lo ejecutó en Colab con los datos reales y revisó las salidas.

Los prompts se transcriben tal como se escribieron. Las preguntas de la conversación que no generaron código no se incluyen.

**Prompt 1 — Punto de partida**
> «tengo este proyecto: [se adjuntó el documento de instrucciones del TP, con el contexto, las fases, el método de corrección de fechas y los valores de control] este es el github del rpoeycto, https://github.com/julia-iza/Trabajo-Practico-Laboratorio. podes inicar cargando los archivos y ese paso?»

*Qué aportó:* el método de corrección de fechas de 2020 y 2021 y los valores de control. → Secciones 9, 10 y 17.

**Prompt 2 — Manejo de los archivos y subida a GitHub**
> «limpia el codigo como quieras, y decime exactamente donde subir los archvios y si van a quedar en el repositorio de git o no y como hacer el push despues»

*Qué aportó:* los zips quedan en Drive y no se suben a GitHub (pesan demasiado). → Sección 2.

**Prompt 3 — Años incluidos**
> «inlcui todos los años en el notebook»

*Qué aportó:* se trabaja con todos los años, de 2015 a 2026. → Sección 1.

**Prompt 4 — Inspección previa**
> «PERFECTO todo salio bien, ahora cual seria el sigueinte paso?»
> «si armalo»

*Qué aportó:* la inspección de los archivos originales. → Parte 1.

**Prompt 5 — Limpieza adaptada a cada año**
> [se adjuntó el reporte de la inspección]
> «y cuanot temrines de hacre seto, decime donde eberia subir el archivo de insepccion apra que el rpofesor vea el analisis y el resultado tambien, y como hacerlo y como subir el arhvio que me vas a pasar ahora»

*Qué aportó:* la limpieza adaptada a lo que mostró la inspección. → Secciones 8 y 9 a 16.

**Prompt 6 — Formato de notebook**
> «tengo un tema, el profesor dijo que si o si tenemos que tener os notebook s en github con formato del collab.»

*Qué aportó:* la limpieza pasó de un script de Python a notebook, con una explicación antes de cada celda.

**Prompt 7 — Controles de calidad**
> «okey perfecto, ahora quiero que me expliques linea y seccion por seccion que hace este notebook, porque, que deberia revisar yo de la salida de eso para hacerle QA y realmente revisar la logica detras»

*Qué aportó:* los controles A a D. → Secciones 18 a 21.

**Prompt 8 — Un solo notebook**
> «creo que me gustaria juntar el notebook de limpieza de datos y de inspeccion de datos, que lo primero que haga es justamente inspeccionarlos, y despues limpiarlos»

*Qué aportó:* la unión de la inspección y la limpieza en este notebook. → Todo el notebook.

**Prompt 9 — Nombres de estaciones dañados**
> [se pegó la salida del control C, que mostraba la misma estación escrita de varias formas: `SAENZ PEÃ±A`, `SAENZ PE�A`, `AGÃ¼ERO`, `RETIRO E`, entre otras]

*Qué aportó:* la reparación de la codificación, la limpieza de símbolos, `CORRECCIONES_ESTACIONES` y la opción `REPROCESAR`. → Secciones 1, 12 y 15.

**Qué revisó el grupo**
- [Ejecutamos el notebook completo con los datos reales y leímos el reporte de la inspección antes de revisar la limpieza.]
- [Revisamos la salida de cada año: filas leídas, filas sin fecha, filas fuera del mes.]
- [Comparamos los controles de la Parte 3 con los valores esperados.]
- [Con el control C encontramos estaciones con nombres dañados, que se corrigieron en la sección 12.]
- (completar)